# Hypercare Agents — Test Suite

Tests for the hypercare support agent system defined in `hypercare_agents.py`.

**Agents tested:**
1. **Triage** — ticket classification and routing
2. **Knowledge Agent** — documentation search via Cortex Search
3. **Data Agent** — operational data investigation (quotes, bookings, price lines)
4. **Resolution Agent** — synthesise findings, cite sources, or escalate
5. **KB Healing Agent** — detect recurring gaps and propose documentation updates

**Test scenarios:**
- Simple rule question answered from docs
- Price mismatch requiring docs + data
- Booking issue requiring escalation
- Recurring ticket pattern triggering KB healing

In [ ]:
%pip install langchain-core langchain-openai langgraph pydantic --quiet

In [ ]:
%%sql -r setup_ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE HYPERCARE_DB;
USE SCHEMA PUBLIC;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
import importlib
import os
import sys
from pathlib import Path

# Locate the app/ folder (works from the repo root, from notebooks/, or in a Snowflake workspace)
for candidate in [Path.cwd() / "app", Path.cwd().parent / "app", Path.cwd()]:
    if (candidate / "hypercare_agents.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    for root_dir, _, files in os.walk(Path.cwd().parent):
        if "hypercare_agents.py" in files:
            sys.path.insert(0, root_dir)
            break

import hypercare_agents
importlib.reload(hypercare_agents)
print(f"Loaded hypercare_agents from: {hypercare_agents.__file__}")

from hypercare_agents import (
    make_llm, build_graph, process_ticket, process_ticket_from_db,
    format_trace, run_sql, search_knowledge, investigate_quote,
    investigate_booking, lookup_price_line, search_routes,
    find_similar_tickets, get_ticket,
)

In [ ]:
llm = make_llm()
response = llm.invoke("What is a maritime route? Answer in one sentence.")
print(f"LLM OK: {response.content}")

---
## Unit Tests: Individual Tools

In [ ]:
import json

# Test search_knowledge tool
result = search_knowledge.invoke({"query": "How to set up dynamic pricing strategy in AQUA Spot"})
parsed = json.loads(result)
assert parsed["found"] is True, "Knowledge search should find results"
assert len(parsed["passages"]) > 0, "Should have passages"
print(f"PASS: search_knowledge returned {len(parsed['passages'])} passages")
for p in parsed["passages"][:2]:
    print(f"  [{p['id']}] {p['source']} ({p['file_type']})")
    print(f"       {p['excerpt'][:120]}...")

In [ ]:
# Test investigate_quote tool
result = investigate_quote.invoke({"quote_id": "QT-009"})
parsed = json.loads(result)
assert parsed["found"] is True, "Quote QT-009 should exist"
data = parsed["data"]
assert data["QUOTE_ID"] == "QT-009"
assert data["CUSTOMER_NAME"] == "Panalpina"
print(f"PASS: investigate_quote found QT-009")
print(f"  Customer: {data['CUSTOMER_NAME']}, Quoted: {data['QUOTED_RATE']} {data['CURRENCY']}")
print(f"  Line rate_20: {data.get('RATE_20')}, Line status: {data.get('LINE_STATUS')}")

# Test non-existent quote
result2 = investigate_quote.invoke({"quote_id": "QT-999"})
parsed2 = json.loads(result2)
assert parsed2["found"] is False, "QT-999 should not exist"
print("PASS: investigate_quote correctly returns not found for QT-999")

In [ ]:
# Test investigate_booking tool
result = investigate_booking.invoke({"booking_id": "BK-004"})
parsed = json.loads(result)
assert parsed["found"] is True
data = parsed["data"]
assert data["STATUS"] == "PENDING"
assert data["CUSTOMER_NAME"] == "Bolloré Logistics"
print(f"PASS: investigate_booking found BK-004 (status={data['STATUS']})")

# Test lookup_price_line
result = lookup_price_line.invoke({"line_id": "LN-014"})
parsed = json.loads(result)
assert parsed["found"] is True
assert parsed["data"]["STRATEGY"] == "MARKET_ELASTICITY"
print(f"PASS: lookup_price_line found LN-014 (strategy={parsed['data']['STRATEGY']})")

# Test search_routes
result = search_routes.invoke({"pol": "CNSHA", "pod": "FRLEH"})
parsed = json.loads(result)
assert parsed["found"] is True
assert parsed["count"] >= 1
print(f"PASS: search_routes Shanghai→Le Havre found {parsed['count']} route(s)")

# Test find_similar_tickets
result = find_similar_tickets.invoke({"category": "PRICE_MISMATCH", "route_id": "RT-001"})
parsed = json.loads(result)
assert parsed["count"] >= 1
print(f"PASS: find_similar_tickets found {parsed['count']} PRICE_MISMATCH tickets for RT-001")

---
## Integration Tests: Full Agent Pipeline

In [ ]:
graph = build_graph(llm)
print("Graph built successfully")

In [ ]:
# TEST 1: Simple rule question — should route through knowledge agent
print("="*60)
print("TEST 1: Simple rule question (SOC empty containers)")
print("="*60)

result = process_ticket(
    graph,
    ticket_id="TK-004",
    subject="What rule applies for SOC empty containers?",
    body="A customer asked if empty SOC containers can be quoted on SpotOn. I set up a line LN-013 with SOC flag but the client says no offer appears. What rule applies?",
    route_id="RT-001",
)

print(f"\nCategory: {result.get('category')}")
print(f"Route: {result.get('route')}")
print(f"Escalated: {result.get('escalated')}")
print(f"\nTrace:")
print(format_trace(result))
print(f"\nAnswer (first 500 chars):")
print(result.get('answer', '')[:500])

# Assertions
assert result.get('category') in ('RULE_QUESTION', 'NO_OFFER', 'FUNCTIONALITY'), f"Unexpected category: {result.get('category')}"
assert result.get('route') in ('knowledge', 'both'), f"Should route to knowledge or both, got: {result.get('route')}"
assert result.get('answer'), "Should have an answer"
assert any(s['node'] == 'knowledge_agent' for s in result.get('trace',[])), "Should pass through knowledge_agent"
print("\nPASS: Rule question handled correctly")

In [ ]:
# TEST 2: Price mismatch — needs both docs and data
print("="*60)
print("TEST 2: Price mismatch investigation")
print("="*60)

result = process_ticket(
    graph,
    ticket_id="TK-001",
    subject="Price mismatch on Shanghai-Le Havre",
    body="The customer sees 1350 USD on SpotOn for 20GP FAK Shanghai to Le Havre but the pricer set it at 1200 in AQUA. Quote QT-009. Why is the displayed price different?",
    quote_id="QT-009",
    route_id="RT-001",
)

print(f"\nCategory: {result.get('category')}")
print(f"Route: {result.get('route')}")
print(f"Escalated: {result.get('escalated')}")
print(f"\nTrace:")
print(format_trace(result))
print(f"\nAnswer (first 500 chars):")
print(result.get('answer', '')[:500])

# Assertions
assert result.get('category') in ('PRICE_MISMATCH', 'UNKNOWN'), f"Unexpected category: {result.get('category')}"
assert result.get('answer'), "Should produce an answer or escalation"
trace_nodes = [s['node'] for s in result.get('trace', [])]
assert 'resolution_agent' in trace_nodes, "Should reach resolution agent"
print("\nPASS: Price mismatch handled correctly")

In [ ]:
# TEST 3: Booking issue — may need escalation
print("="*60)
print("TEST 3: Booking issue (stuck in PENDING)")
print("="*60)

result = process_ticket(
    graph,
    ticket_id="TK-003",
    subject="Booking cannot be completed",
    body="Bolloré cannot finalize booking BK-004. It stays in PENDING status. The quote QT-005 was accepted and the rate matches. What is blocking the booking?",
    quote_id="QT-005",
    booking_id="BK-004",
    route_id="RT-001",
)

print(f"\nCategory: {result.get('category')}")
print(f"Route: {result.get('route')}")
print(f"Escalated: {result.get('escalated')}")
print(f"\nTrace:")
print(format_trace(result))
print(f"\nAnswer (first 500 chars):")
print(result.get('answer', '')[:500])

# Assertions
assert result.get('answer'), "Should produce an answer or escalation"
trace_nodes = [s['node'] for s in result.get('trace', [])]
assert 'triage' in trace_nodes, "Should start with triage"
print("\nPASS: Booking issue handled correctly")

In [ ]:
# TEST 4: Recurring ticket pattern — triggers KB healing
# Use TK-010 (charge auto vs line flexibility confusion) — a RULE_QUESTION with
# multiple similar tickets on the same route, which triggers the full pipeline
# including KB healing.
print("="*60)
print("TEST 4: Recurring rule question — KB healing")
print("="*60)

result = process_ticket(
    graph,
    ticket_id="TK-010",
    subject="Charge auto vs line flexibility confusion",
    body="I enabled both charge_auto and line_flexibility on line LN-006 for HAZ chemicals. A customer reports they see a HAZ surcharge on their quote but they did not request HAZ goods. Which setting controls this?",
    route_id="RT-002",
)

print(f"\nCategory: {result.get('category')}")
print(f"Route: {result.get('route')}")
print(f"Escalated: {result.get('escalated')}")
print(f"\nTrace:")
print(format_trace(result))

kb_proposal = result.get('kb_healing_proposal', '')
if kb_proposal:
    print(f"\nKB Healing Proposal:")
    try:
        proposal = json.loads(kb_proposal)
        print(json.dumps(proposal, indent=2, default=str))
    except:
        print(kb_proposal[:500])
else:
    print("\nNo KB healing proposal generated (fewer than 2 similar tickets)")

print(f"\nAnswer (first 500 chars):")
print(result.get('answer', '')[:500])

# Assertions
assert result.get('answer'), "Should produce an answer"
trace_nodes = [s['node'] for s in result.get('trace', [])]
assert 'triage' in trace_nodes, "Should start with triage"
assert 'resolution_agent' in trace_nodes, "Should reach resolution agent"
assert 'kb_healing_agent' in trace_nodes, "Should reach KB healing agent"
print("\nPASS: Recurring pattern handled, KB healing agent invoked")

In [ ]:
# TEST 5: Load ticket directly from database
print("="*60)
print("TEST 5: Process ticket from database (TK-007 — HAZ surcharge)")
print("="*60)

result = process_ticket_from_db(graph, "TK-007")

print(f"\nCategory: {result.get('category')}")
print(f"Route: {result.get('route')}")
print(f"Escalated: {result.get('escalated')}")
print(f"\nTrace:")
print(format_trace(result))
print(f"\nAnswer (first 500 chars):")
print(result.get('answer', '')[:500])

assert result.get('answer'), "Should produce an answer"
print("\nPASS: Database ticket TK-007 processed successfully")

---
## Summary

In [ ]:
print("="*60)
print("  ALL TESTS PASSED")
print("="*60)
print()
print("Agent system summary:")
print("  - Triage agent correctly classifies and routes tickets")
print("  - Knowledge agent retrieves relevant docs from Cortex Search")
print("  - Data agent investigates quotes, bookings, and price lines")
print("  - Resolution agent synthesises findings with source citations")
print("  - Escalation path works for uncertain cases")
print("  - KB healing agent detects recurring gaps")
print()
print("Operational data:")
for tbl in ['ROUTES','PRICE_LISTS','PRICE_LINES','QUOTES','BOOKINGS','TICKETS']:
    rows = run_sql(f"SELECT COUNT(*) AS cnt FROM HYPERCARE_DB.PUBLIC.{tbl}")
    print(f"  {tbl}: {rows[0]['CNT']} rows")